# 05 - Meta-analysis

Collects the per-study results written by `04_launch_gwas.ipynb`, runs the
meta-analysis (PLINK for a quick fixed/random-effects pass, METAL via the
pipeline for the reported results), and checks the output with Manhattan and
QQ plots plus per-study genomic inflation factors.

A "study" here is one study arm of one dataset: the pipeline writes one
`*_y_allresults.tsv` per arm, so an ancestry with several cohorts contributes
several files to a survival meta-analysis.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gwas import config as cfg
from gwas import meta
from gwas import plots as gp

pd.options.mode.chained_assignment = None

cfg.describe()

In [ ]:
# Mount the release and working buckets with gcsfuse (run once per VM).
for cmd in cfg.mount_commands():
    print(cmd)

## Collect the per-study results

In [ ]:
ANALYSIS = 'SURV'            # 'SURV' or 'CS_disc'
ANCESTRIES = ['EUR', 'AJ']   # None takes every ancestry folder present
MODALITIES = [meta.NBA, meta.WGS]

paths = meta.collect_result_paths(ANALYSIS, ancestries=ANCESTRIES,
                                  modalities=MODALITIES)

In [ ]:
# Drop any study that should not go into this meta-analysis, by label.
EXCLUDE = {
    # 'EUR_PDGEN': 'too few samples, unstable estimates',
}

for label, reason in EXCLUDE.items():
    print(f'excluding {label}: {reason}')

paths = [p for p in paths if meta.study_label(p) not in EXCLUDE]
print(f'\n{len(paths)} studies going into the {ANALYSIS} meta-analysis')

## Per-study QC

Check every study before combining them: a lambda far from 1, or a study with
no usable p-values, is a problem to fix rather than to average away.

In [ ]:
lambdas = gp.qq_by_study(meta.labelled_paths(paths),
                         title=f'Per-study QQ - {ANALYSIS}',
                         save_path=f'{ANALYSIS}_qq_by_study.png')
plt.show()
display(lambdas)

## PLINK meta-analysis

A quick fixed- and random-effects pass that also reports Cochran's Q and
I-squared, which the forest plots reuse.

In [ ]:
label = f'{ANALYSIS}_{"_".join(ANCESTRIES)}'
out_dir = cfg.prep_path('meta', 'PLINK', ANALYSIS)
out_dir.mkdir(parents=True, exist_ok=True)

print(meta.plink_meta_command(paths, out_dir / label))

In [ ]:
plink_meta = pd.read_csv(out_dir / f'{label}.meta', sep=r'\s+')
print(f'{len(plink_meta)} variants')

# I is I-squared in percent; above 80 the studies disagree too much
plink_meta = plink_meta[plink_meta.I < 80]
print(f'{len(plink_meta)} variants with I2 < 80')

display(plink_meta[plink_meta.P == plink_meta.P.min()])
display(plink_meta[plink_meta.P < gp.GENOME_WIDE])

## Manhattan and QQ

In [ ]:
gp.manhattan_plot(plink_meta,
                  title=f'{"_".join(ANCESTRIES)} meta-analysis - {ANALYSIS}',
                  save_path=f'{label}_manhattan.png')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
lam = gp.qq_plot(plink_meta['P'], ax,
                 title=f'{"_".join(ANCESTRIES)} meta - {ANALYSIS}')
ax.set_xlabel(r'Expected $-\log_{10}(P)$')
ax.set_ylabel(r'Observed $-\log_{10}(P)$')
plt.tight_layout()
plt.savefig(f'{label}_qq.png', dpi=300)
plt.show()

print(f'lambda = {lam:.4f}')

## METAL

The reported meta-analysis. `run_metal.nf` reads the same per-study files; on
the cloud profile it needs gs:// paths, so the local mount paths are
translated first.

In [ ]:
gcs_paths = [cfg.as_gcs(p) for p in paths]

metal_outdir = cfg.prep_url('meta', 'METAL', ANALYSIS)
print(meta.metal_command(gcs_paths, metal_outdir, f'HY_{ANALYSIS}_META',
                         profile='gcb_gcp'))

In [ ]:
# Running locally instead: copy the inputs to local disk first and use the
# standard profile with local paths.
print(meta.metal_command(paths, cfg.prep_path('meta', 'METAL', ANALYSIS),
                         f'HY_{ANALYSIS}_META', profile='standard'))

## Compare PLINK and METAL

Sanity check to make sure the two should agree closely; variants where they do not usually point at an
allele-alignment problem in one of the inputs.

In [ ]:
metal = pd.read_csv(cfg.prep_path('meta', 'METAL', ANALYSIS,
                                  f'HY_{ANALYSIS}_META.TBL'), sep='\t')
print(f'{len(metal)} variants, {len(metal.columns)} columns')

combined = plink_meta.merge(metal, left_on='SNP', right_on='MarkerName', how='inner')
print(f'{len(combined)} variants in both')

disagree = combined[np.abs(combined.P - combined['Pvalue']) > 0.01]
print(f'{len(disagree)} variants where the p-values differ by more than 0.01')
display(disagree.head())

In [ ]:
out = cfg.prep_path('meta', 'combined', ANALYSIS)
out.mkdir(parents=True, exist_ok=True)
combined.to_csv(out / f'{label}.csv', index=False)
print(f'wrote {out / f"{label}.csv"}')

Next: `06_clean_results.ipynb` annotates the METAL output for GWASlab and
LocusZoom; `07_forest_plots.ipynb` and `08_beta_beta_plots.ipynb` cover the
per-variant follow-up.